In [1]:
import numpy as np
import pickle
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [17]:
# load citeseer data
data = np.load("citeseer_prepared.npz", allow_pickle=True)

X = data["X"]
y = data["y"]
train_indices = data["train_indices"]
test_indices = data["test_indices"]
node_ids = data["node_ids"]

In [18]:
# load neighbor dictionary
with open("citeseer_neighbors.pkl", "rb") as f:
    neighbors = pickle.load(f)

In [5]:
# recreate node to index mapping
node_to_index = {
    node_id: index
    for index, node_id in enumerate(node_ids)
}

# check the shapes
print("Features:", X.shape)
print("Labels:", y.shape)
print("Train nodes:", len(train_indices))
print("Test nodes:", len(test_indices))
print("Number of neighbor sets:", len(neighbors))

Features: (3312, 3703)
Labels: (3312,)
Train nodes: 2649
Test nodes: 663
Number of neighbor sets: 3312


In [7]:
# get node data
def get_node_data(node_id):
    center_index = node_to_index[node_id]

    center_features = X[center_index]

    neighbor_ids = list(neighbors[node_id])

    neighbor_features = np.array([
        X[node_to_index[neighbor_id]]
        for neighbor_id in neighbor_ids
    ])

    label = y[center_index]

    return center_features, neighbor_features, label

# test node data
center, neighbor_features, label = get_node_data(node_ids[0])

print("Center shape:", center.shape)
print("Neighbors shape:", neighbor_features.shape)
print("Label:", label)

Center shape: (3703,)
Neighbors shape: (12, 3703)
Label: 1


In [8]:
# create dataset
class CiteseerDataset(Dataset):
    def __init__(self, indices):
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        node_index = self.indices[idx]
        node_id = node_ids[node_index]

        center_features, neighbor_features, label = get_node_data(node_id)

        return center_features, neighbor_features, label

In [9]:
# create train and test datasets
train_dataset = CiteseerDataset(train_indices)
test_dataset = CiteseerDataset(test_indices)

print("Training samples:", len(train_dataset))
print("Test samples:", len(test_dataset))

Training samples: 2649
Test samples: 663


In [10]:
# create collate function
def citeseer_collate(batch):
    centers = []
    neighbor_features = []
    labels = []

    for center, neighbors, label in batch:
        centers.append(
            torch.tensor(center, dtype=torch.float32)
        )

        neighbor_features.append(
            torch.tensor(neighbors, dtype=torch.float32)
        )

        labels.append(label)

    centers = torch.stack(centers)
    labels = torch.tensor(labels, dtype=torch.long)

    return centers, neighbor_features, labels

In [11]:
# create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=citeseer_collate
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=citeseer_collate
)

In [12]:
# test one batch
centers, neighbor_features, labels = next(iter(train_loader))

print("Centers:", centers.shape)
print("Number of neighbor sets:", len(neighbor_features))

for i, neighbors in enumerate(neighbor_features[:4]):
    print(f"Neighbor set {i + 1}:", neighbors.shape)

print("Labels:", labels.shape)

Centers: torch.Size([16, 3703])
Number of neighbor sets: 16
Neighbor set 1: torch.Size([1, 3703])
Neighbor set 2: torch.Size([3, 3703])
Neighbor set 3: torch.Size([4, 3703])
Neighbor set 4: torch.Size([1, 3703])
Labels: torch.Size([16])


In [13]:
# define the model
class CiteseerSetTransformer(nn.Module):
    def __init__(
        self,
        input_dim=3703,
        hidden_dim=64,
        num_heads=4,
        num_classes=6
    ):
        super().__init__()

        self.input_projection = nn.Linear(
            input_dim,
            hidden_dim
        )

        self.self_attention = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=num_heads,
            batch_first=True
        )

        self.center_embedding = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU()
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim + hidden_dim, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, center, neighbors):
        neighbors = self.input_projection(neighbors)

        attended, _ = self.self_attention(
            neighbors,
            neighbors,
            neighbors
        )

        pooled = attended.mean(dim=0)

        center_features = self.center_embedding(center)

        combined = torch.cat(
            [center_features, pooled],
            dim=0
        )

        output = self.classifier(combined)

        return output

In [14]:
# create the model
model = CiteseerSetTransformer()

print(model)

CiteseerSetTransformer(
  (input_projection): Linear(in_features=3703, out_features=64, bias=True)
  (self_attention): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
  )
  (center_embedding): Sequential(
    (0): Linear(in_features=3703, out_features=64, bias=True)
    (1): ReLU()
  )
  (classifier): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=6, bias=True)
  )
)


In [19]:
# test one forward pass
center, neighbor_features, label = train_dataset[0]

center = torch.tensor(center, dtype=torch.float32)
neighbor_features = torch.tensor(
    neighbor_features,
    dtype=torch.float32
)

output = model(center, neighbor_features)

print("Center:", center.shape)
print("Neighbors:", neighbor_features.shape)
print("Output:", output.shape)
print("True label:", label)

Center: torch.Size([3703])
Neighbors: torch.Size([1, 3703])
Output: torch.Size([6])
True label: 0


In [20]:
# count trainable parameters
total_params = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("Trainable parameters:", total_params)

Trainable parameters: 499398


In [21]:
# loss function and optimizer
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [22]:
# train set transformer
num_epochs = 30

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0

    for centers, neighbor_features, labels in train_loader:
        optimizer.zero_grad()

        batch_outputs = []

        for i in range(len(centers)):
            output = model(
                centers[i],
                neighbor_features[i]
            )

            batch_outputs.append(output)

        outputs = torch.stack(batch_outputs)

        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch + 1:02d}/{num_epochs}, "
        f"Loss: {average_loss:.4f}"
    )

Epoch 01/30, Loss: 1.0899
Epoch 02/30, Loss: 0.4286
Epoch 03/30, Loss: 0.1326
Epoch 04/30, Loss: 0.0463
Epoch 05/30, Loss: 0.0228
Epoch 06/30, Loss: 0.0218
Epoch 07/30, Loss: 0.0140
Epoch 08/30, Loss: 0.0063
Epoch 09/30, Loss: 0.0068
Epoch 10/30, Loss: 0.0053
Epoch 11/30, Loss: 0.0030
Epoch 12/30, Loss: 0.0026
Epoch 13/30, Loss: 0.0020
Epoch 14/30, Loss: 0.0012
Epoch 15/30, Loss: 0.0025
Epoch 16/30, Loss: 0.0019
Epoch 17/30, Loss: 0.0011
Epoch 18/30, Loss: 0.0020
Epoch 19/30, Loss: 0.0008
Epoch 20/30, Loss: 0.0043
Epoch 21/30, Loss: 0.0008
Epoch 22/30, Loss: 0.0002
Epoch 23/30, Loss: 0.0001
Epoch 24/30, Loss: 0.0001
Epoch 25/30, Loss: 0.0001
Epoch 26/30, Loss: 0.0001
Epoch 27/30, Loss: 0.0001
Epoch 28/30, Loss: 0.0000
Epoch 29/30, Loss: 0.0000
Epoch 30/30, Loss: 0.0000


In [23]:
# evaluate accuracy

model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():
    for centers, neighbor_features, labels in test_loader:

        batch_outputs = []

        for i in range(len(centers)):
            output = model(
                centers[i],
                neighbor_features[i]
            )

            batch_outputs.append(output)

        outputs = torch.stack(batch_outputs)

        predictions = torch.argmax(outputs, dim=1)

        all_predictions.extend(predictions.numpy())
        all_labels.extend(labels.numpy())

accuracy = accuracy_score(
    all_labels,
    all_predictions
)

print(f"Test accuracy: {accuracy:.4f}")

Test accuracy: 0.7557


In [24]:
# confusion matrix
cm = confusion_matrix(
    all_labels,
    all_predictions
)

print(cm)

[[ 19   6   7   3   2  13]
 [  5 104   3   4   1   2]
 [  2   4 111   2  14   7]
 [  0  11   1  79   8   3]
 [  1   4   9   5 100  15]
 [  7   3   6   2  12  88]]


In [25]:
# classification report
print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=[
            "AI",
            "Agents",
            "DB",
            "HCI",
            "IR",
            "ML"
        ]
    )
)

              precision    recall  f1-score   support

          AI       0.56      0.38      0.45        50
      Agents       0.79      0.87      0.83       119
          DB       0.81      0.79      0.80       140
         HCI       0.83      0.77      0.80       102
          IR       0.73      0.75      0.74       134
          ML       0.69      0.75      0.72       118

    accuracy                           0.76       663
   macro avg       0.73      0.72      0.72       663
weighted avg       0.75      0.76      0.75       663



In [26]:
# test one random permutation

model.eval()

center, neighbor_features, label = test_dataset[0]

center = torch.tensor(center, dtype=torch.float32)
neighbor_features = torch.tensor(
    neighbor_features,
    dtype=torch.float32
)

with torch.no_grad():
    original_output = model(
        center,
        neighbor_features
    )

    original_probability = torch.softmax(
        original_output,
        dim=0
    )

    permutation = torch.randperm(
        neighbor_features.size(0)
    )

    shuffled_neighbors = neighbor_features[permutation]

    shuffled_output = model(
        center,
        shuffled_neighbors
    )

    shuffled_probability = torch.softmax(
        shuffled_output,
        dim=0
    )

difference = torch.abs(
    original_probability - shuffled_probability
)

print("Original probabilities:", original_probability)
print("Shuffled probabilities:", shuffled_probability)
print("Maximum difference:", difference.max().item())

Original probabilities: tensor([2.9356e-05, 6.6208e-09, 2.8325e-08, 2.4374e-07, 9.9997e-01, 7.1892e-07])
Shuffled probabilities: tensor([2.9356e-05, 6.6208e-09, 2.8325e-08, 2.4374e-07, 9.9997e-01, 7.1892e-07])
Maximum difference: 0.0


In [27]:
# test 10 random permutations
differences = []

model.eval()

with torch.no_grad():
    original_output = model(
        center,
        neighbor_features
    )

    original_probability = torch.softmax(
        original_output,
        dim=0
    )

    for _ in range(10):
        permutation = torch.randperm(
            neighbor_features.size(0)
        )

        shuffled_neighbors = neighbor_features[permutation]

        shuffled_output = model(
            center,
            shuffled_neighbors
        )

        shuffled_probability = torch.softmax(
            shuffled_output,
            dim=0
        )

        max_difference = torch.abs(
            original_probability - shuffled_probability
        ).max().item()

        differences.append(max_difference)

print("Differences:", differences)
print("Maximum difference across all permutations:", max(differences))

Differences: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Maximum difference across all permutations: 0.0
